# AR Rehabilitation RL Controller Simulation
## Experiment 2 — RL Validation & Comparative Study

### Abstract & Research Objectives
Unilateral Spatial Neglect (USN) stroke rehabilitation requires adaptive target placement to continuously challenge the patient's neglected visual field without causing frustration, $30\text{s}$ timeouts, or cognitive fatigue.

In **Experiment 2**, we validate the Reinforcement Learning contribution by comparing 6 target placement strategies:

| Method | Type | Adaptive? |
| :--- | :--- | :---: |
| **Fixed** | Static Moderate Difficulty | ❌ |
| **Random** | Unadapted Uniform Choice | ❌ |
| **Rule-based** | Heuristic Step Controller | ✓ |
| **PPO** | Deep Reinforcement Learning | ✓ |
| **DQN** | Deep Reinforcement Learning | ✓ |
| **A2C** | Deep Reinforcement Learning | ✓ |

### Performance Metrics Measured:
1. **Cumulative Reward**: Accumulated session reward.
2. **Success Rate (%)**: Percentage of successful non-timeout hits.
3. **Difficulty Progression**: Tracking target parameter adjustments (`speed`, `eccentricity_deg`, `distance_m`, `time_limit_s`).
4. **Timeout Rate (%)**: Percentage of trials resulting in $30\text{s}$ timeouts.
5. **Adaptation Stability**: Variance/standard deviation of difficulty adjustments (smooth adaptation vs staircasing).

In [ ]:
# Setup & Core Package Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO, DQN, A2C

from unity_schema import UnitySession, UnityTrial, DifficultyAtTrial
from unity_env import UnityARRehabEnv
from unity_dataset import session_to_observation, predict_next_difficulty
from utils import set_seeds, BenchmarkPolicies, RuleBasedPolicy
from visualization import plot_experiment2_benchmark, plot_difficulty_progression_comparison, plot_learning_curves

set_seeds(42)
print("Environment & Setup Successfully Initialized!")

### 1. Phase 1 — Pre-training RL Algorithms (PPO, DQN, A2C)
We train PPO, DQN, and A2C agents on `UnityARRehabEnv` for 30,000 steps to learn optimal 4-action trial difficulty strategies.

In [ ]:
from train_unity import train_unity_agent

rl_results = {}
for algo in ["PPO", "DQN", "A2C"]:
    model, df_logs = train_unity_agent(algo_name=algo, total_timesteps=30000, seed=42)
    rl_results[algo] = df_logs

print("Phase 1 RL Pre-training Complete for PPO, DQN, A2C!")

### 2. Experiment 2 — RL Validation Benchmark Execution
We benchmark simulated patients across 50 test episodes for Fixed, Random, Rule-based, PPO, DQN, and A2C methods.

In [ ]:
from evaluate import run_experiment2_validation

summary_df, progression_dict = run_experiment2_validation(num_episodes=50, seed=42)
display(summary_df)

### 3. Display Research Visualizations & Comparison Charts

In [ ]:
from IPython.display import Image, display
from config import OUTPUT_DIR

exp2_bar_img = os.path.join(OUTPUT_DIR, "exp2_benchmark_comparison.png")
exp2_traj_img = os.path.join(OUTPUT_DIR, "exp2_difficulty_progression.png")
rl_curves_img = os.path.join(OUTPUT_DIR, "learning_curves_comparison.png")

print("Figure 1: Experiment 2 — Method Benchmark Comparison Bar Charts")
if os.path.exists(exp2_bar_img):
    display(Image(filename=exp2_bar_img))

print("Figure 2: Trial Difficulty Progression Trajectory (Rule-Based Staircasing vs PPO Smooth Adaptation)")
if os.path.exists(exp2_traj_img):
    display(Image(filename=exp2_traj_img))

print("Figure 3: 3-RL Algorithm Learning Curves Comparison (PPO vs DQN vs A2C)")
if os.path.exists(rl_curves_img):
    display(Image(filename=rl_curves_img))